# Лабораторная работа 6. Анализ частотности слов и N-грамм в корпусах текстов на узбекском языке

**Технологии больших данных (Big Data)**, 6 семестр.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OWNER/REPO/blob/main/notebooks/l06.ipynb)

**Цели занятия:**

- Построить частотный словарь корпуса и проверить закон Ципфа (ранг × частота, наклон в логарифмических осях)
- Сравнить частотность слов по категориям и выделить ключевые слова категорий методом TF-IDF
- Найти устойчивые словосочетания (коллокации) с помощью PMI
- Исследовать агглютинативную морфологию: частоты суффиксов и самые частые корни после стемминга

> Автоматическая проверка заданий доступна в веб-тренажёре. В блокноте выполняйте код и сверяйте результат с условием.

In [ ]:
# ▶ Подготовка среды: выполните эту ячейку первой.
# Загружает учебные данные из репозитория тренажёра и запускает локальный веб-сервер
# (нужен для заданий с requests: учебный сайт SITE и учебный API).
import os, sys, subprocess, time

REPO = "https://github.com/OWNER/REPO.git"
if not os.path.exists("data/manifest.json"):
    if "OWNER/REPO" in REPO:
        raise SystemExit("Укажите в REPO адрес репозитория тренажёра на GitHub, например https://github.com/<логин>/bigdata-trainer.git")
    if not os.path.exists("bigdata-trainer"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, "bigdata-trainer"], check=True)
    os.chdir("bigdata-trainer")

subprocess.Popen([sys.executable, "-m", "http.server", "8765"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)
BASE_URL = "http://localhost:8765/"
SITE = BASE_URL + "data/site/"
API = BASE_URL + "data/api/v1/"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openpyxl", "wordcloud", "seaborn"])
print("Готово. Рабочая папка:", os.getcwd())

## Теория

## Частотный словарь и закон Ципфа

**Частотный словарь** — список слов корпуса, упорядоченный по убыванию частоты. Номер слова в этом списке — его **ранг** *r* (1 — самое частое слово).

Американский лингвист Дж. Ципф заметил, что частота слова обратно пропорциональна его рангу:

```
f(r) ≈ C / r^a,   a ≈ 1      →      log f = log C − a · log r
```

Отсюда два практических следствия:

- произведение **ранг × частота** примерно постоянно для слов «средней» частоты;
- в логарифмических осях (log r, log f) точки ложатся почти на **прямую** с наклоном ≈ −1. Наклон находят линейной регрессией:

```python
slope, intercept = np.polyfit(np.log10(freq_df["rank"]), np.log10(freq_df["count"]), 1)
plt.loglog(freq_df["rank"], freq_df["count"], ".")
```

Закон Ципфа объясняет, почему Big Data в лингвистике неизбежна: несколько сотен самых частых слов покрывают половину любого текста, а остальной словарь — это «длинный хвост» редких слов, и чтобы увидеть каждое из них хотя бы несколько раз, нужны миллионы токенов.

> **Заметка: Маленький корпус — «плоский хвост»**  
> В нашем корпусе (~1 600 токенов) больше трёх четвертей словаря — **гапаксы** (частота 1). Они образуют горизонтальную «ступеньку» в конце графика и делают наклон более пологим. Поэтому наклон часто оценивают только по словам с частотой ≥ 2.

## Частотность по категориям и ключевые слова

Длинную таблицу «один токен — одна строка» удобно получить методом `explode`, а матрицу «слово × категория» — функцией `pd.crosstab`:

```python
tok_df = df[["category", "tokens"]].explode("tokens").rename(columns={"tokens": "word"})
cat_counts = pd.crosstab(tok_df["word"], tok_df["category"])
```

**Ключевые слова** категории — слова, частые в ней и редкие в остальных. Два популярных способа:

| Метод | Формула | Идея |
|---|---|---|
| отношение относительных частот | `(f_кат / N_кат) / (f_корп / N_корп)` | во сколько раз слово «чаще обычного» |
| **TF-IDF** | `tf(слово, док) × idf(слово)`, `idf = ln((1 + n) / (1 + df)) + 1` | частое в документе, но встречается в немногих документах |

```python
from sklearn.feature_extraction.text import TfidfVectorizer
vec = TfidfVectorizer(stop_words=STOP_LIST)       # по умолчанию: lowercase, L2-нормировка
X = vec.fit_transform(cat_docs)                    # строки — категории, столбцы — слова
tfidf = pd.DataFrame(X.toarray(), index=cat_docs.index, columns=vec.get_feature_names_out())
tfidf.loc["sport"].nlargest(5)
```

Шаблон токенов sklearn по умолчанию `(?u)\b\w\w+\b` правильно работает с узбекскими словами: знаки ʻ и ʼ — буквы, поэтому *oʻzbek* остаётся одним токеном.

## Коллокации и PMI

**Коллокация** — устойчивое сочетание, слова которого встречаются вместе чаще, чем случайно: *sunʼiy intellekt*, *terma jamoa*. Самые частые биграммы — не обязательно коллокации (*tili uchun* частая, но случайная). Мера **PMI** (pointwise mutual information) сравнивает реальную частоту пары с ожидаемой при независимости слов:

```
PMI(x, y) = log2( P(x, y) / (P(x) · P(y)) ) ≈ log2( c(x, y) · N / (c(x) · c(y)) )
```

PMI = 0 — слова независимы, PMI > 0 — притягиваются. Недостаток: у редких пар PMI завышен (пара из двух гапаксов, встретившаяся один раз, получает максимальный PMI). Поэтому вводят **порог частоты** (например, `c(x, y) ≥ 2`) и отбрасывают пары со стоп-словами. Более устойчивые меры — t-score и log-likelihood (`nltk.collocations.BigramAssocMeasures`).

## Морфология агглютинативного языка в цифрах

В узбекском грамматические значения выражаются цепочками суффиксов, поэтому частоты **суффиксов** многое говорят о текстах:

| Суффикс | Значение | Пример |
|---|---|---|
| -lar | множественное число | talaba**lar** |
| -ning | родительный падеж | talabalar**ning** |
| -ni | винительный падеж | kitob**ni** |
| -ga | дательный падеж | maktab**ga** |
| -da | местный падеж | Toshkent**da** |
| -dan | исходный падеж | yil**dan** |
| -dagi | признак по месту | shahar**dagi** |
| -lik, -chi | словообразование: свойство, деятель | kitobxon**lik**, dastur**chi** |
| -moqda, -gan, -di | время глагола | rivojlan**moqda**, boʻl**gan**, ochil**di** |

Простой подсчёт «слово оканчивается на суффикс» даёт приблизительную картину: *qanday* не содержит суффикса -da, хотя оканчивается на «-day». Для надёжной оценки нужны морфологический анализатор и словарь корней. **Стемминг** объединяет словоформы одного корня (*yil, yilda, yildan, yilga, yilning* → *yil*), и частотный список основ показывает **темы** корпуса точнее, чем список словоформ.

## Задание 1. Частотный словарь корпуса

В `df["tokens"]` — токены каждого документа (нормализованы, в нижнем регистре).

1. `freq` — `Counter` по всем токенам корпуса.
2. `freq_df` — DataFrame со столбцами `word`, `count`, отсортированный по убыванию частоты, и столбцом `rank` = 1, 2, 3, … (номер строки).
3. `hapax_share` — доля слов словаря с частотой 1.
4. `coverage_100` — доля всех токенов, которую покрывают 100 самых частых слов.
5. `n_for_half` — **минимальное** число самых частых слов, которые вместе покрывают не меньше 50 % токенов (подсказка: `cumsum`).

<details><summary>💡 Подсказка</summary>

`freq.most_common()` без аргумента возвращает все слова по убыванию. Ранг: `np.arange(1, len(freq_df) + 1)`. Минимальное число слов: `int((cum_share < 0.5).sum()) + 1`.

</details>

In [ ]:
# Подготовительный код задания
import re
import numpy as np
import pandas as pd
from collections import Counter

def normalize(text):
    """Все варианты апострофа -> канонические: oʻ/gʻ (U+02BB) и ʼ (U+02BC)."""
    text = re.sub(r"([oOgG])['\u2018\u2019\u0060]", lambda m: m.group(1) + "\u02bb", text)
    return re.sub(r"['\u2018\u2019\u0060]", "\u02bc", text)

def tokenize(text):
    return re.findall(r"\w+", normalize(text).lower())

df = pd.read_csv("data/corpus_uz/meta.csv")[["file", "title", "category"]]
df["text"] = [open("data/corpus_uz/" + f, encoding="utf-8").read() for f in df["file"]]
df["tokens"] = df["text"].apply(tokenize)
with open("data/stopwords_uz.txt", encoding="utf-8") as _f:
    STOPWORDS = {w.strip() for w in _f if w.strip()}

In [ ]:
freq = ...
freq_df = pd.DataFrame(..., columns=["word", "count"])
freq_df["rank"] = ...

hapax_share = ...
coverage_100 = ...

cum_share = freq_df["count"].cumsum() / freq_df["count"].sum()
n_for_half = ...

print(f"Словарь: {len(freq_df)} слов, гапаксов {hapax_share:.1%}")
print(f"Топ-100 покрывают {coverage_100:.1%} токенов; половину текста дают {n_for_half} слов")
freq_df.head(15)

## Задание 2. Проверка закона Ципфа

Таблица `freq_df` (word, count, rank) уже построена; `rank` — номер строки в отсортированном списке.

1. Добавьте столбец `rf` = rank × count и посмотрите на него для рангов 1–20 и 50–100: постоянно ли произведение?
2. Найдите наклон прямой в логарифмических осях: `slope_all` — `np.polyfit(np.log10(rank), np.log10(count), 1)[0]` по **всем** словам; `slope_top` — то же только для слов с частотой **≥ 2**.
3. Постройте график частоты от ранга в логарифмических осях (`plt.loglog` или `set_xscale("log")` и `set_yscale("log")`) и добавьте на него прямую регрессии для `slope_top`.

<details><summary>💡 Подсказка</summary>

`np.polyfit(np.log10(freq_df["rank"]), np.log10(freq_df["count"]), 1)` возвращает `[наклон, сдвиг]`. Прямая: `ax.loglog(top["rank"], 10 ** (b_top + slope_top * np.log10(top["rank"])))`.

</details>

In [ ]:
# Подготовительный код задания
import re
import numpy as np
import pandas as pd
from collections import Counter

def normalize(text):
    """Все варианты апострофа -> канонические: oʻ/gʻ (U+02BB) и ʼ (U+02BC)."""
    text = re.sub(r"([oOgG])['\u2018\u2019\u0060]", lambda m: m.group(1) + "\u02bb", text)
    return re.sub(r"['\u2018\u2019\u0060]", "\u02bc", text)

def tokenize(text):
    return re.findall(r"\w+", normalize(text).lower())

df = pd.read_csv("data/corpus_uz/meta.csv")[["file", "title", "category"]]
df["text"] = [open("data/corpus_uz/" + f, encoding="utf-8").read() for f in df["file"]]
df["tokens"] = df["text"].apply(tokenize)
with open("data/stopwords_uz.txt", encoding="utf-8") as _f:
    STOPWORDS = {w.strip() for w in _f if w.strip()}

freq = Counter(t for toks in df["tokens"] for t in toks)
freq_df = pd.DataFrame(freq.most_common(), columns=["word", "count"])
freq_df["rank"] = np.arange(1, len(freq_df) + 1)

In [ ]:
import matplotlib.pyplot as plt

freq_df["rf"] = ...
print(freq_df.iloc[list(range(0, 20)) + list(range(49, 100)), :].to_string(max_rows=30))

slope_all, b_all = np.polyfit(..., ..., 1)
top = freq_df[freq_df["count"] >= 2]
slope_top, b_top = ...

print(f"Наклон по всем словам: {slope_all:.2f}, по словам с частотой ≥ 2: {slope_top:.2f}")

fig, ax = plt.subplots(figsize=(7, 5))
...
plt.show()

## Задание 3. Частотность по категориям

Используйте метаданные (`category`), чтобы сравнить лексику рубрик.

1. `tok_df` — длинная таблица со столбцами `category`, `file`, `word`: одна строка — один токен **без стоп-слов** (метод `explode`).
2. `cat_counts` — матрица «слово × категория» с частотами: `pd.crosstab(tok_df["word"], tok_df["category"])`.
3. `spread` — Series: в скольких категориях встречается каждое слово.
4. `universal` — отсортированный список слов, которые встречаются **не меньше чем в 4** категориях. Это общая новостная лексика, а не тематическая.

<details><summary>💡 Подсказка</summary>

Фильтр: `tok_df[~tok_df["word"].isin(STOPWORDS) & tok_df["word"].notna()]`. Число категорий: `(cat_counts > 0).sum(axis=1)`. Список: `sorted(spread[spread >= 4].index)`.

</details>

In [ ]:
# Подготовительный код задания
import re
import numpy as np
import pandas as pd
from collections import Counter

def normalize(text):
    """Все варианты апострофа -> канонические: oʻ/gʻ (U+02BB) и ʼ (U+02BC)."""
    text = re.sub(r"([oOgG])['\u2018\u2019\u0060]", lambda m: m.group(1) + "\u02bb", text)
    return re.sub(r"['\u2018\u2019\u0060]", "\u02bc", text)

def tokenize(text):
    return re.findall(r"\w+", normalize(text).lower())

df = pd.read_csv("data/corpus_uz/meta.csv")[["file", "title", "category"]]
df["text"] = [open("data/corpus_uz/" + f, encoding="utf-8").read() for f in df["file"]]
df["tokens"] = df["text"].apply(tokenize)
with open("data/stopwords_uz.txt", encoding="utf-8") as _f:
    STOPWORDS = {w.strip() for w in _f if w.strip()}

In [ ]:
tok_df = df[["category", "file", "tokens"]].explode("tokens").rename(columns={"tokens": "word"})
tok_df = ...          # убрать стоп-слова (и пустые значения)

cat_counts = ...
spread = ...
universal = ...

print(cat_counts.shape)
print(universal)
cat_counts.loc[["talabalar", "yangi", "katta"]]

## Задание 4. Ключевые слова категорий: TF-IDF

Найдите ключевые слова каждой рубрики методом TF-IDF.

1. `cat_docs` — Series: индекс — категория, значение — **все тексты категории**, нормализованные (`normalize`), в нижнем регистре и склеенные через пробел. Получится 12 «документов».
2. `TfidfVectorizer(stop_words=STOP_LIST)` с остальными параметрами по умолчанию: обучите на `cat_docs` и постройте DataFrame `tfidf` (строки — категории в порядке `cat_docs.index`, столбцы — слова, `get_feature_names_out()`).
3. `keywords` — словарь `{категория: список 5 слов с наибольшим TF-IDF}`.

<details><summary>💡 Подсказка</summary>

Склейка: `lambda texts: " ".join(normalize(t).lower() for t in texts)`. Обучение: `X = vec.fit_transform(cat_docs)`; слова: `vec.get_feature_names_out()`. Топ-5: `tfidf.loc[cat].nlargest(5).index.tolist()`.

</details>

In [ ]:
# Подготовительный код задания
import re
import numpy as np
import pandas as pd
from collections import Counter

def normalize(text):
    """Все варианты апострофа -> канонические: oʻ/gʻ (U+02BB) и ʼ (U+02BC)."""
    text = re.sub(r"([oOgG])['\u2018\u2019\u0060]", lambda m: m.group(1) + "\u02bb", text)
    return re.sub(r"['\u2018\u2019\u0060]", "\u02bc", text)

def tokenize(text):
    return re.findall(r"\w+", normalize(text).lower())

df = pd.read_csv("data/corpus_uz/meta.csv")[["file", "title", "category"]]
df["text"] = [open("data/corpus_uz/" + f, encoding="utf-8").read() for f in df["file"]]
df["tokens"] = df["text"].apply(tokenize)
with open("data/stopwords_uz.txt", encoding="utf-8") as _f:
    STOPWORDS = {w.strip() for w in _f if w.strip()}

STOP_LIST = sorted(w for w in STOPWORDS if "-" not in w)   # без слов с дефисом (ham-da)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

cat_docs = df.groupby("category")["text"].apply(lambda texts: ...)

vec = ...
X = ...
tfidf = pd.DataFrame(X.toarray(), index=cat_docs.index, columns=...)

keywords = {cat: ... for cat in tfidf.index}

for cat, words in keywords.items():
    print(f"{cat:12} {', '.join(words)}")

## Задание 5. Коллокации: PMI биграмм

В `sent_tokens` — списки токенов каждого предложения корпуса, в `freq` — частоты слов, `N` — число токенов.

1. `bigram_freq` — `Counter` биграмм **внутри предложений**.
2. Для биграмм с частотой **≥ 2**, в которых **нет стоп-слов**, посчитайте
   `PMI = log2( c(x, y) · N / (c(x) · c(y)) )`, где `c(x)`, `c(y)` — частоты слов из `freq`.
3. `pmi_df` — DataFrame со столбцами `w1`, `w2`, `count`, `pmi`, отсортированный по `pmi` по убыванию.
4. Сравните: какие пары в начале списка по PMI и какие — по частоте?

<details><summary>💡 Подсказка</summary>

Биграммы предложения: `bigram_freq.update(zip(toks, toks[1:]))`. Условие: `c >= 2 and w1 not in STOPWORDS and w2 not in STOPWORDS`. PMI: `np.log2(c * N / (freq[w1] * freq[w2]))`. Сортировка: `pd.DataFrame(rows).sort_values("pmi", ascending=False)`.

</details>

In [ ]:
# Подготовительный код задания
import re
import numpy as np
import pandas as pd
from collections import Counter

def normalize(text):
    """Все варианты апострофа -> канонические: oʻ/gʻ (U+02BB) и ʼ (U+02BC)."""
    text = re.sub(r"([oOgG])['\u2018\u2019\u0060]", lambda m: m.group(1) + "\u02bb", text)
    return re.sub(r"['\u2018\u2019\u0060]", "\u02bc", text)

def tokenize(text):
    return re.findall(r"\w+", normalize(text).lower())

df = pd.read_csv("data/corpus_uz/meta.csv")[["file", "title", "category"]]
df["text"] = [open("data/corpus_uz/" + f, encoding="utf-8").read() for f in df["file"]]
df["tokens"] = df["text"].apply(tokenize)
with open("data/stopwords_uz.txt", encoding="utf-8") as _f:
    STOPWORDS = {w.strip() for w in _f if w.strip()}

def split_sentences(text):
    return [s for s in re.split(r"(?<=[.!?])\s+|\n+", text) if s.strip()]

sent_tokens = [tokenize(s) for text in df["text"] for s in split_sentences(text)]
freq = Counter(t for toks in df["tokens"] for t in toks)
N = sum(freq.values())

In [ ]:
bigram_freq = Counter()
for toks in sent_tokens:
    ...

rows = []
for (w1, w2), c in bigram_freq.items():
    if ...:
        pmi = ...
        rows.append({"w1": w1, "w2": w2, "count": c, "pmi": pmi})

pmi_df = ...

print("По частоте:", pmi_df.sort_values("count", ascending=False).head(5)[["w1", "w2", "count"]].values.tolist())
pmi_df.head(15)

## Задание 6. Частоты суффиксов агглютинативной морфологии

В `content_tokens` — токены корпуса без стоп-слов, в `SUFFIXES` — список суффиксов.

1. `suffix_freq` — Series (индекс — суффикс): сколько токенов (с повторами) **оканчивается** на этот суффикс так, что перед ним остаётся **не меньше 3 символов** (`t.endswith(suf) and len(t) - len(suf) >= 3`). Одно слово может засчитываться нескольким суффиксам. Отсортируйте по убыванию.
2. `plural_share` — доля токенов, в которых показатель множественного числа «lar» стоит **не в начале** слова: `t.find("lar", 3) != -1` (учитываются и *talabalar*, и *talabalarning*).
3. `top_case` — самый частый падежный суффикс из `["ning", "ni", "ga", "da", "dan"]` (строка).
4. Постройте столбчатую диаграмму `suffix_freq`.

<details><summary>💡 Подсказка</summary>

Счёт: `sum(1 for t in content_tokens if t.endswith(suf) and len(t) - len(suf) >= 3)`. Сортировка: `.sort_values(ascending=False)`. Падеж: `suffix_freq[["ning", "ni", "ga", "da", "dan"]].idxmax()`.

</details>

In [ ]:
# Подготовительный код задания
import re
import numpy as np
import pandas as pd
from collections import Counter

def normalize(text):
    """Все варианты апострофа -> канонические: oʻ/gʻ (U+02BB) и ʼ (U+02BC)."""
    text = re.sub(r"([oOgG])['\u2018\u2019\u0060]", lambda m: m.group(1) + "\u02bb", text)
    return re.sub(r"['\u2018\u2019\u0060]", "\u02bc", text)

def tokenize(text):
    return re.findall(r"\w+", normalize(text).lower())

df = pd.read_csv("data/corpus_uz/meta.csv")[["file", "title", "category"]]
df["text"] = [open("data/corpus_uz/" + f, encoding="utf-8").read() for f in df["file"]]
df["tokens"] = df["text"].apply(tokenize)
with open("data/stopwords_uz.txt", encoding="utf-8") as _f:
    STOPWORDS = {w.strip() for w in _f if w.strip()}

content_tokens = [t for toks in df["tokens"] for t in toks if t not in STOPWORDS]

SUFFIXES = ["lar", "ning", "ni", "ga", "da", "dan", "dagi", "lik", "chi", "moqda", "gan", "di", "ish", "si"]

In [ ]:
import matplotlib.pyplot as plt

suffix_freq = pd.Series({suf: ... for suf in SUFFIXES})
suffix_freq = ...

plural_share = ...
top_case = ...

...
print(f"Множественное число (-lar): {plural_share:.1%} токенов; самый частый падеж: -{top_case}")
suffix_freq

## Задание 7. Самые частые корни после стемминга

Функция `stem(word)` (стеммер из П12) и `content_tokens` уже готовы.

1. `stem_freq` — `Counter` основ всех `content_tokens`.
2. `top_stems` — 10 самых частых основ (`most_common(10)`).
3. `forms` — словарь `{основа: отсортированный список разных словоформ корпуса с этой основой}` для каждой основы из `top_stems`.
4. `stripped_share` — доля токенов, у которых стеммер что-то отрезал (`stem(t) != t`): насколько «суффиксален» текст.
5. Сравните `top_stems` с 10 самыми частыми **словоформами**: какие темы корпуса стали заметнее?

<details><summary>💡 Подсказка</summary>

`Counter(stem(t) for t in content_tokens)`. Формы: `sorted({w for w in content_tokens if stem(w) == s})`. Доля: `sum(stem(t) != t for t in content_tokens) / len(content_tokens)`.

</details>

In [ ]:
# Подготовительный код задания
import re
import numpy as np
import pandas as pd
from collections import Counter

def normalize(text):
    """Все варианты апострофа -> канонические: oʻ/gʻ (U+02BB) и ʼ (U+02BC)."""
    text = re.sub(r"([oOgG])['\u2018\u2019\u0060]", lambda m: m.group(1) + "\u02bb", text)
    return re.sub(r"['\u2018\u2019\u0060]", "\u02bc", text)

def tokenize(text):
    return re.findall(r"\w+", normalize(text).lower())

df = pd.read_csv("data/corpus_uz/meta.csv")[["file", "title", "category"]]
df["text"] = [open("data/corpus_uz/" + f, encoding="utf-8").read() for f in df["file"]]
df["tokens"] = df["text"].apply(tokenize)
with open("data/stopwords_uz.txt", encoding="utf-8") as _f:
    STOPWORDS = {w.strip() for w in _f if w.strip()}

content_tokens = [t for toks in df["tokens"] for t in toks if t not in STOPWORDS]

SUFFIXES = ["lar", "ning", "ni", "ga", "da", "dan", "dagi", "lik", "chi", "si",
            "imiz", "ingiz", "moqda", "gan", "di", "adi", "ish", "ib"]

def stem(word):
    """Стеммер из П12: отрезает самый длинный суффикс, пока основа ≥ 3 символов."""
    changed = True
    while changed:
        changed = False
        for suf in sorted(SUFFIXES, key=len, reverse=True):
            if word.endswith(suf) and len(word) - len(suf) >= 3:
                word, changed = word[:-len(suf)], True
                break
    return word

In [ ]:
stem_freq = ...
top_stems = ...
forms = {}
for s, c in top_stems:
    forms[s] = ...

stripped_share = ...

print("Словоформы:", Counter(content_tokens).most_common(10))
print("Основы:    ", top_stems)
for s, f in forms.items():
    print(f"{s:12} {f}")
print(f"Стеммер изменил {stripped_share:.1%} токенов")

## Самостоятельная работа

**Тема для самостоятельной работы: «Перспективы применения технологий Big Data в компьютерной лингвистике».**

1. Соберите узбекский корпус объёмом не менее 1 млн токенов (Википедия, новости) и повторите анализ: закон Ципфа, ключевые слова, коллокации. Как изменился наклон и доля гапаксов?
2. Сравните коллокации по PMI, t-score и log-likelihood (`nltk.collocations`). Какая мера даёт более осмысленные сочетания?
3. Подготовьте доклад: какие задачи узбекской компьютерной лингвистики (машинный перевод, поиск, распознавание речи) упираются в объём данных и как Big Data-технологии помогают их решать.